# Alzheimer's Disease Classification — Optimized Model

This notebook implements an **optimized CNN** for classifying brain MRI images into four Alzheimer's Disease (AD) stages using **EfficientNetB0** transfer learning.

### Classes
1. MildDemented
2. ModerateDemented
3. NonDemented
4. VeryMildDemented

### Dataset
Google Drive: https://drive.google.com/drive/folders/1mpwG9EZBcD-tyJs9Fj1Rkq0O6eFQRQVX

## 1. Install & Import Dependencies

In [ ]:
# Install/upgrade required packages
!pip install -q tensorflow scikit-learn matplotlib seaborn Pillow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Mount Google Drive & Configure Dataset Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -----------------------------------------------------------------------
# Update DATASET_PATH to point to the folder that contains the 4 class
# subdirectories (MildDemented, ModerateDemented, NonDemented,
# VeryMildDemented) inside your Google Drive.
# -----------------------------------------------------------------------
DATASET_PATH = '/content/drive/MyDrive/AD_Dataset'  # <-- adjust if needed

CLASS_NAMES   = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
SEED          = 42

# Verify that the expected class directories exist
for cls in CLASS_NAMES:
    path = os.path.join(DATASET_PATH, cls)
    count = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f'  {cls}: {count} images')

## 3. Data Preprocessing & Augmentation

In [ ]:
# Training generator — with heavy augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,          # 80% train, 20% validation
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest',
)

# Validation/test generator — only rescale
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=SEED,
    shuffle=True,
)

val_gen = val_test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=SEED,
    shuffle=False,
)

print(f'Train samples     : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Class indices     : {train_gen.class_indices}')

In [ ]:
# Visualise a batch of augmented training images
images, labels = next(train_gen)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(CLASS_NAMES[np.argmax(labels[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Augmented Training Images', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Compute Class Weights (handles class imbalance)

In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASS_NAMES)),
    y=train_gen.classes,
)
class_weights = dict(enumerate(class_weights_array))
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()})

## 5. Build the Transfer Learning Model (EfficientNetB0)

In [ ]:
def build_model(num_classes: int = 4, img_size: tuple = (224, 224), dropout_rate: float = 0.4) -> Model:
    """Build an EfficientNetB0-based transfer learning model."""
    inputs = keras.Input(shape=(*img_size, 3))

    # --- Base model (frozen) ---
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
    )
    base_model.trainable = False  # Freeze for initial training

    # --- Classification head ---
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(dropout_rate, name='dropout_1')(x)
    x = layers.BatchNormalization(name='bn_256')(x)
    x = layers.Dense(128, activation='relu', name='dense_128')(x)
    x = layers.Dropout(dropout_rate / 2, name='dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs, outputs, name='AD_EfficientNetB0')
    return model, base_model


model, base_model = build_model(num_classes=len(CLASS_NAMES))
model.summary()

## 6. Phase 1 — Train Classification Head (frozen base)

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/AD_Models'
os.makedirs(SAVE_DIR, exist_ok=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=os.path.join(SAVE_DIR, 'ad_phase1_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

print('--- Phase 1: Training classification head (base frozen) ---')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1,
)

## 7. Phase 2 — Fine-tune Top Layers of EfficientNetB0

In [ ]:
# Unfreeze the top 30 layers of the base model for fine-tuning
base_model.trainable = True
FINE_TUNE_FROM = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

trainable_count = sum(1 for layer in model.layers if layer.trainable)
print(f'Trainable layers after unfreezing top 30: {trainable_count}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=os.path.join(SAVE_DIR, 'ad_best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

print('--- Phase 2: Fine-tuning top layers ---')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1,
)

## 8. Plot Training History

In [ ]:
def merge_histories(h1, h2):
    """Concatenate two Keras history objects."""
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined


history = merge_histories(history1, history2)
epochs_range = range(1, len(history['accuracy']) + 1)
phase1_end   = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epochs_range, history['accuracy'],     label='Train Accuracy')
axes[0].plot(epochs_range, history['val_accuracy'], label='Val Accuracy')
axes[0].axvline(phase1_end, color='grey', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(epochs_range, history['loss'],     label='Train Loss')
axes[1].plot(epochs_range, history['val_loss'], label='Val Loss')
axes[1].axvline(phase1_end, color='grey', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Training History (Phase 1 + Phase 2 Fine-tuning)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=150)
plt.show()

## 9. Evaluate on Validation Set

In [ ]:
# Reset the generator before prediction to ensure correct ordering
val_gen.reset()

val_loss, val_acc = model.evaluate(val_gen, verbose=1)
print(f'\nValidation Loss    : {val_loss:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}')

# Predictions
val_gen.reset()
y_pred_proba = model.predict(val_gen, verbose=1)
y_pred  = np.argmax(y_pred_proba, axis=1)
y_true  = val_gen.classes

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30)
ax.set_title('Confusion Matrix — Validation Set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 11. Classification Report

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print('Classification Report:')
print(report)

## 12. Sample Predictions with Actual vs Predicted Labels

In [ ]:
val_gen.reset()
sample_images, sample_labels = next(val_gen)
sample_preds = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, ax in enumerate(axes.flat):
    if i >= len(sample_images):
        ax.axis('off')
        continue
    ax.imshow(sample_images[i])
    actual    = CLASS_NAMES[np.argmax(sample_labels[i])]
    predicted = CLASS_NAMES[np.argmax(sample_preds[i])]
    confidence = np.max(sample_preds[i]) * 100
    color = 'green' if actual == predicted else 'red'
    ax.set_title(
        f'A: {actual}\nP: {predicted} ({confidence:.1f}%)',
        fontsize=8,
        color=color,
    )
    ax.axis('off')

plt.suptitle('Sample Predictions  (Green = correct, Red = incorrect)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sample_predictions.png'), dpi=150)
plt.show()

## 13. Save Final Model

In [ ]:
final_model_path = os.path.join(SAVE_DIR, 'ad_final_model.keras')
model.save(final_model_path)
print(f'Model saved to: {final_model_path}')

# Also export as SavedModel format for broader compatibility
saved_model_path = os.path.join(SAVE_DIR, 'ad_saved_model')
model.export(saved_model_path)
print(f'SavedModel exported to: {saved_model_path}')

## 14. (Optional) Load & Verify the Saved Model

In [ ]:
loaded_model = keras.models.load_model(final_model_path)
loaded_model.summary()

# Quick sanity check on a single batch
val_gen.reset()
imgs, lbls = next(val_gen)
preds = loaded_model.predict(imgs, verbose=0)
print('Loaded model predictions shape:', preds.shape)
print('All predictions sum to 1 (softmax check):', np.allclose(preds.sum(axis=1), 1.0))